# Extracting and analyzing spike-times from membrane potential recording

### Project outline 

Extract spike times from an electrophysiological recordings and characterize spiking statistics. You are provided with whole-cell recordings from medium spiny neurons in the striatum which received pre-synaptic inputs and current injections. The direct current injection elicited action potenials in the recorded neurons. The recordings was used to elicit spike-timing dependent plasticity. The aim is to extract the times of the presynaptic stimulation - from the stimulus artifact - and the times of the postsynaptic action potentials. Firing rates, the coefficient of variation and the time difference between the pre-synaptic stimulation and the postsynaptic action potentials are to be calculated. In other words, compute the cross-correlogram between pre-and postsynaptic activity.

### Related literature 

The recordings stem from cortico-striatal plasticity experiments performed in the lab of Laurent Venance of the College de France. An introductory read about those experiments would be [Paille et al. 2013](Paille2013_JNeurosci.pdf).

<div class="alert alert-info"> 
Using the information provided, try to learn as much as possible about the scientific and technical background of the project. 
    
* have a in depth look at the publication provided with the project (if any), here : [Paille et al. 2013](Paille2013_JNeurosci.pdf).
* search for project keywords, here : "spike-timing dependent plasticity", "whole-cell recording", "coefficient of variation", "cross-correlation"
* if applicable : look back at the lecture and turorial which dealt with the technique to be used in the project, here : "Lecture 4 : Wrangling spike trains : basic analysis, raster plot, PSTH, Poisson spike trains"
 </span>

### Read and display the raw data

<div class="alert alert-info"> Try to first understand in detail the structure of the data. What is contained in the data structure and what are the properties of the data. </span>

In [7]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal
%matplotlib qt


data1 = np.load('20160303_1.1_time-voltage.npy')
data2 = np.load('20170221_005_time-voltage.npy')

print(np.shape(data1))
print(np.shape(data2))


(1202000, 2)
(351200, 2)


In [8]:
data1

array([[ 0.00000000e+00, -7.13125000e-02],
       [ 1.00000083e-04, -7.08125000e-02],
       [ 2.00000166e-04, -7.07812500e-02],
       ...,
       [ 1.20199800e+02, -6.30000000e-02],
       [ 1.20199900e+02, -6.31562500e-02],
       [ 1.20200000e+02, -6.33125000e-02]])

<div class="alert alert-info">As always, plotting the data provide immediate and useful insights. Generally, plot data and results helps to develop a better understanding. I would recommend plotting in an external window, since this allows to zoom into the data. This is in particular useful for large data-sets. Don't forget to include figures in the jupyter-notebook for the project presentation and for illustrating the results obtained. 
    
 * zoom into the data and explore features at many different time points
 * try to get a first rough estimate about the project question through visual inspection, here : what is the firing rate of the neuron 
 * if data-set is too large, plot and inspect smaller sections of the data 
</span>

In [9]:
# plot the data 
fig = plt.figure(figsize=(10,10))
ax0 = fig.add_subplot(211)
ax0.plot(data1[:,0],data1[:,1]*1000.)
ax0.set_xlabel('time (s)')
ax0.set_ylabel('potential (mV)')

ax1 = fig.add_subplot(212)
ax1.plot(data2[:,0],data2[:,1]*1000.)
ax1.set_xlabel('time (s)')
ax1.set_ylabel('potential (mV)')

plt.show()

### Analysis : How to extract the times of peaks (or troughs)

The function `scipy.signal.find_peaks()` seems suited to extract the timing of local peaks in a time series. See [this website](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.find_peaks.html) for more details. 

<div class="alert alert-info">Adapt the code for the project. In other words, start from the scratch, internet, tutorial code and adapt it  
    
* Change the values and fill in the project specifics. 
* It is useful and good practise to separate **computations**: the actual implementation of the analysis and **visualization** : the figure generation. 
</span>


<div class="alert alert-info">Sometimes, the analysis can take minutes or even hours, when the data-set is large or the computations heavy. Be patient. You can try to let the analysis run over night.</div>

**Figure generation**
Show your results in figure with captions and labels.

<div class="alert alert-info">Producing good representations of results in form of figures takes time and effort. Try to find the visualization which best conveys your result. Certain measures have standard representations (e.g. frequency content is shown in spectrograms, spikes in raster plots, etc.).</div>

### Example code to calculate cross-correlogram 

In the code below, the pre- and postsynaptic spike time are constructed. In the loop all time differences between pre- and postsynaptic spike times are calculated. The histogram of those time differences gives information about the delay between pre- and postsynpatic spikes. 

In [5]:
tpre   = np.random.random(100)*8
tpost = tpre + 0.035

In [6]:
tpre

array([2.90703519, 6.98984696, 2.32657244, 0.98996853, 0.57054229,
       2.45145549, 4.3834267 , 4.07627667, 1.34873363, 5.1801596 ,
       2.67220006, 5.17863564, 3.81932928, 5.46579981, 1.7479451 ,
       5.90345417, 5.23165257, 6.42704652, 0.31925522, 4.47022985,
       3.04811333, 7.96093123, 7.94225805, 1.35908573, 3.58538887,
       2.39640439, 2.17779572, 4.94628855, 7.21418774, 4.67864339,
       7.06371963, 1.68041007, 0.57906931, 6.49792993, 7.32924187,
       6.60741723, 0.56971012, 5.18552733, 4.97410563, 2.34654829,
       4.94239253, 6.99442175, 0.72882658, 2.06343297, 4.34661911,
       3.02565463, 3.1923531 , 5.37965165, 7.37005269, 3.63955492,
       2.46345113, 2.9842214 , 0.54736338, 2.89590675, 7.47708472,
       1.52602875, 0.66078057, 2.51033228, 0.34103279, 3.18894688,
       6.23599967, 1.19943347, 3.59055328, 6.78328234, 3.79609932,
       5.15158219, 6.09857019, 4.98616585, 1.0704981 , 4.69257507,
       0.9294903 , 1.24082944, 6.45320263, 3.70920626, 6.33384

In [7]:
tpost

array([2.94203519, 7.02484696, 2.36157244, 1.02496853, 0.60554229,
       2.48645549, 4.4184267 , 4.11127667, 1.38373363, 5.2151596 ,
       2.70720006, 5.21363564, 3.85432928, 5.50079981, 1.7829451 ,
       5.93845417, 5.26665257, 6.46204652, 0.35425522, 4.50522985,
       3.08311333, 7.99593123, 7.97725805, 1.39408573, 3.62038887,
       2.43140439, 2.21279572, 4.98128855, 7.24918774, 4.71364339,
       7.09871963, 1.71541007, 0.61406931, 6.53292993, 7.36424187,
       6.64241723, 0.60471012, 5.22052733, 5.00910563, 2.38154829,
       4.97739253, 7.02942175, 0.76382658, 2.09843297, 4.38161911,
       3.06065463, 3.2273531 , 5.41465165, 7.40505269, 3.67455492,
       2.49845113, 3.0192214 , 0.58236338, 2.93090675, 7.51208472,
       1.56102875, 0.69578057, 2.54533228, 0.37603279, 3.22394688,
       6.27099967, 1.23443347, 3.62555328, 6.81828234, 3.83109932,
       5.18658219, 6.13357019, 5.02116585, 1.1054981 , 4.72757507,
       0.9644903 , 1.27582944, 6.48820263, 3.74420626, 6.36884

In [8]:
allDeltaTs = []
for i in np.arange(len(tpre)):
    deltaT = tpost - tpre[i]
    allDeltaTs.extend(deltaT)

In [9]:
len(allDeltaTs)

10000

**Further analysis and interpretation of results**

The specific objective for this project was to find frequency ranges which exhibit dynamics over time. Here, it looks like as if theta power is present only for certain time intervals. Let's extract the mean power in the theta range and display it over time.

<div class="alert alert-info">Going an additional step after obtaining the main result provides often great insight, doesn't take much effort and increases the quality/impact of your work.</div>

In [11]:
plt.hist(allDeltaTs,bins=1000)
plt.show()

**Discussion**

<div class="alert alert-info">Finish with a disucssion which contains a summary of your results and an interpretation. What does it mean? What would be further steps?</div>